# Simple Single Lens (with ARC) simulation with MeepSAT

`NOTE`: First, make sure to connect to the correctly installed version of MeepSAT's jupyter kernel 

In this tutorial, we will simulate a 2D Gaussian beam propagating through a plano-convex lens with anti-reflective (AR) coatings. By the end of this tutorial, you will understand how to:

- Configure simulation parameters in JSON format
- Define a Gaussian-pulsed source (so that frequency-domain fields can be extracted at the centre frequency with DFT monitors)
- Define simple optical components like lenses and apertures
- Visualize and analyze the electromagnetic field evolution over time.
- Extract the frequency-domain $\textbf{E}$ and $\textbf{H}$ fields at the source frequency using MEEP's DFT monitors and compute the Poynting vector $\textbf{S}$
- Compute the far-field beam with MEEP's built-in near-to-far field (N2F) transformation and compare it with the current industry standard softwares such as GRASP, CST

In [ ]:
# Importing various Python libraries and MeepSAT modules
import sys
import os
import site
from pathlib import Path
import meep as mp
import numpy as np
import h5py
import matplotlib.pyplot as plt
import time
import json

# Importing the MEEPSAT librarires
import meepsat.simulator as sim
import meepsat.meep_geometry as comp_meep
import meepsat.permittivity_components as comp_eps
import meepsat.stepfunctions as stepfunctions
import meepsat.json_to_script as json_to_script
import meepsat.field_analysis as mpsat_analysis
import meepsat.helpers as mpsat_helpers

# JSON file path representing mainly the different optical components parameters
json_file_path = 'auxiliary_data/01_simple_single_lens_ARC/simple_single_lens_ARC.json'
data = mpsat_helpers.read_json(json_file_path)

# Some universal constants
c_mm_s = 299792458.0 * 1000.0  # Speed of light in mm/s (m/s -> mm/s)
# Frequency of the simulation
freq = 150.0  # Frequency in GHz
a = 1  # 1 meep unit = 1 mm
wvl = c_mm_s / (freq * 1e9)  # Wavelength in mm
freq_meep = 1.0 / (wvl * a)
print("freq (meep units):", freq_meep)

# Beam waist of the Gaussian beam (in mm)
beam_waist = 1.1660

# Edit the freq in the JSON file 
data["sources"]["source1"]["frequecy"] = freq_meep
data["sources"]["source1"]["extra_args"]["width"] = beam_waist

# Savepath: For storing the output generated during the simulation
savepath = f'auxiliary_data/01_simple_single_lens_ARC/output_files/{freq}GHz'
os.makedirs(savepath, exist_ok=True)
data["output"]["savepath"]["path"] = savepath

Initialising the MeepSAT simulation object from the parameters stored in the JSON file

In [ ]:
# Initialising MEEPSAT Simulation
cell_X, cell_Y, cell_Z = data["simulation"]['primary_params']['cell_size']['x'], data["simulation"]['primary_params']['cell_size']['y'], data["simulation"]['primary_params']['cell_size']['z'] # Cell Size without considering the PML thickness and its factor


# Initialize the simulation with the different parameters
mpsat_sim = sim.sim_init(sim_name= str(data["simulation"]["name"]),
                        cell_size= [cell_X, cell_Y, cell_Z], # [sx, sy, sz] in mm
                        smallest_freq= data["simulation"]['primary_params']['smallest_freq'], 
                        resolution= data["simulation"]['primary_params']['resolution'],
                        boundary_layer_type= data['boundary_layers']['boundary']['type'],
                        boundary_layer_size= data['boundary_layers']['boundary']['size'],
                        factor_dpml= data['boundary_layers']['boundary']['factor_dpml'])


Before creating the components, its very important to check if the mentioned resulution and PML boundary layer thickness is enough for our simulation OR not. In Meep FDTD, its recommended to have atleast 8-10 pixels for the smallest wavelength OR length scale present in your system. 

You can check the resolution and verify using `sim.check_resolution_and_pml`

In [ ]:
# Checking resolution and PML thickness 
# This function will automatically check all the length scales and wavelength scales
data, mpsat_sim = sim.check_resolution_and_pml(
    data=data, 
    mpsat_sim=mpsat_sim,
    smallest_freq=data["simulation"]['primary_params']['smallest_freq'],
    highest_n=data["lenses"]["lens1"]["n_refr"]
)

# Print the simulation parameters
print("\nMEEPSAT SIMULATION PARAMETERS:")
mpsat_sim.print_simulation_parameters()


Now let's add the Source 
- You can either follow the Source documentation mentioned in the ReadTheDocs documentation page

    OR

- Just use MeepSAT's built-in function to generate the source from the JSON file

The JSON file defines a CW (continuous-wave) time profile. For the near-to-far field (N2F) approach used in this notebook, we swap that CW time profile for a **Gaussian pulse** centred at the source frequency: the DFT monitors (near-field line + full-cell field maps) then accumulate the frequency-domain fields at exactly `fcen` while the pulse propagates through and rings down. `GaussianBeam2DSource` evaluates its spatial beam profile at the frequency of its time profile when the source is added, so replacing `src` is sufficient.

In [ ]:
source_list = []

# Swap the CW time profile from the JSON for a Gaussian pulse centred at the
# source frequency; the DFT monitors added before the run extract the fields at
# exactly fcen. GaussianBeam2DSource evaluates its spatial beam profile at the
# frequency of its time profile when the source is added, so replacing src is
# sufficient.
fcen = float(data["sources"]["source1"]["frequecy"])
pulse_rel_bandwidth = 0.2  # Gaussian pulse fwidth as a fraction of fcen
pulse_fwidth = pulse_rel_bandwidth * fcen
for src in source_list:
    src.src = mp.GaussianSource(frequency=fcen, fwidth=pulse_fwidth)
print(f"Gaussian pulse source: fcen = {fcen:.4f} (1/mm), fwidth = {pulse_fwidth:.4f} (1/mm)")

Adding PML boundaries using MEEP

In [ ]:
x_left_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.Low)
x_right_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.X, side=mp.High)
y_down_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.Low)
y_up_boundary = mp.PML(thickness=mpsat_sim.dpml*mpsat_sim.factor_dpml, direction=mp.Y, side=mp.High)

custom_boundary_layers = [x_left_boundary, x_right_boundary, y_down_boundary, y_up_boundary]

Now as we need to add a lot of complex structures (lenses, absorbers etc), we will define a empty epsilon map for this purpose. Its basically a 2D spatial discretization array of the simulation domain and the idea is to draw structure on this 2D array

In [ ]:
size_x, size_y, size_z = mpsat_sim.cell_size[0], mpsat_sim.cell_size[1], mpsat_sim.cell_size[2]
res = int(mpsat_sim.resolution)  # Ensure resolution is an integer
# Create the epsilon map: total size of the simulation cell in all the axis multiplied by the resolution + 1
epsilon_map = np.ones((int((size_x)*res+1), 
                       int((size_y)*res+1)), dtype = 'float32')

Now as we did for the Source, we will use MeepSAT built in function for defining lenses and aperture

In [ ]:
# Adding lens (if given)
exec(json_to_script.add_lens(data))

# Adding aperture (if given)
exec(json_to_script.add_aperture(data))

The system is mirror-symmetric about the y=0 plane. The CW version of this tutorial exploited this with `mp.Mirror(mp.Y, phase=+1)`, but for the pulsed N2F run we follow the reference N2F script and run **without** symmetries (and with real fields): the near-field DFT line monitor and the full-cell DFT field maps are then accumulated on the full grid, which avoids any subtlety with symmetry unfolding of the DFT arrays.

In [ ]:
# Not using mirror symmetry for the pulsed N2F run (see the note above)
symmetries = []

Now defining the Meep Simulation Object

In [ ]:
simulation = mp.Simulation(
    cell_size=mpsat_sim.cell,
    sources=source_list,
    resolution=mpsat_sim.resolution,
    boundary_layers=custom_boundary_layers,
    geometry=mpsat_sim.meep_geometry,
    epsilon_input_file = data["output"]["savepath"]["path"] + data["output"]["epsilon_h5_file"]["filename"] +"_epsilon_map" + ".h5",
    symmetries = symmetries,
    force_complex_fields= False)  # real fields: the DFT monitors provide the complex fields at fcen

simulation.use_output_directory(savepath)

Let's run the simulation briefly to store the epsilon map and visualise the permittivity map

In [ ]:
sim.plot_and_save_epsilon(
    simulation=simulation,
    savepath=savepath + "/",
    filename_prefix="geometry_plot",
    epsilon_data_name="epsilon",
    size_x=size_x,
    size_y=size_y,
    vmin=0.5,
    vmax=3,
    cmap='viridis',
    figsize=(8, 4),
    dpi=300,
    show_plot= True
)

Now let's set the different run time parameters:
- Animation

    `stepfunctions.set_animation_params(...)`
    - Configures how the simulation will be visualized as an animation
        - `image_every`: Frequency of field snapshots (e.g., every N timesteps)
        - `Nfps`: Frames per second for the output video
        - `anim_file_name`: Output path and filename for the MP4 movie

- Field Parameters

    `stepfunctions.set_field_params(...)`
    - Defines spatial and storage parameters for electromagnetic field data
        - `size_x`, `size_y`: Physical dimensions of the simulation domain
        - `savepath`: Directory where field data will be stored
        - `downsampling_factor_x/y`: Reduces data resolution for storage (e.g., keep every Nth point)

- Runtime Parameters

    `runtime_params = sim.calculate_runtime_parameters(...)`
    - Computes temporal simulation parameters based on the source frequency

For the pulsed N2F run, the CW steady-state / time-averaging machinery is no longer used to extract the fields. Instead, the run continues **after the sources turn off** until every DFT monitor (the near-field line and the full-cell field maps) has converged (`mp.stop_when_dft_decayed`), with a hard cap at `2 * runtime_params["total_time"]`. `runtime_params` is still computed to provide the animation timestep and that cap.

In [ ]:
# Set the stepfunctions parameters
# Animation Parameters
stepfunctions.set_animation_params(anim_params= {'image_every': data["output"]["animation_options"]["image_every"], 
                                              'Nfps': data["output"]["animation_options"]["Nfps"], 
                                              'anim_file_name': savepath + "/"+ data["output"]["animation_options"]["movie_name"] + ".mp4"})
# Field Parameters
stepfunctions.set_field_params(field_params= {'size_x': size_x,
                                              'size_y': size_y,
                                              'savepath': savepath,
                                              'downsampling_factor_x': data["output"]["animation_options"]["downsample_x"],
                                              'downsampling_factor_y': data["output"]["animation_options"]["downsample_y"],
                                              'method': 'dft'})

# # Runtime parameters
runtime = 600

runtime_params = sim.calculate_runtime_parameters(
    source_freq=float(data["sources"]["source1"]["frequecy"]),
    resolution= mpsat_sim.resolution,
    steady_state_time = runtime,
    courant=simulation.Courant,
    min_periods_for_steady_state=10,
    periods_to_average=4,
    points_per_period=10,
    animation_timestep=data["output"]["animation_options"]["image_every"])


### MEEP native near-to-far field (N2F) monitor and DFT field maps

Following the [MEEP N2F tutorial](https://meep.readthedocs.io/en/latest/Python_Tutorials/Near_to_Far_Field_Spectra/), a DFT near-field **line monitor** is registered with `add_near2far()` **before** `simulation.run()` so that MEEP can accumulate the Fourier-transformed E and H fields on the line while timestepping. The beam exits the lens/aperture towards $-x$, so the line sits near the left edge of the cell (just outside the PML) and its outward normal is $-x$ (hence `weight = -1`).

We also register full-cell DFT field maps of ($E_z$, $H_x$, $H_y$) at `fcen`: for a pulsed run these replace the CW time-averaged field maps used previously (the instantaneous fields are ~0 once the pulse has left the domain).

In [ ]:
# =================== MEEP native near-to-far field (N2F) monitor ===================
# https://meep.readthedocs.io/en/latest/Python_Tutorials/Near_to_Far_Field_Spectra/
# The DFT near-field monitor must be registered BEFORE simulation.run() so that
# MEEP can accumulate the Fourier-transformed E and H fields on the line while
# timestepping. The beam exits the lens/aperture towards -x, so the line sits near
# the left edge of the cell and its outward normal is -x (hence weight = -1).
pml_thickness = mpsat_sim.dpml * mpsat_sim.factor_dpml

# === far-field (N2F) analysis parameters === #
near_field_x_mm = data["apertures"]["aperture1"]["pos_x"]   # x position of the N2F near-field line monitor in mm
near_field_diameter = data["apertures"]["aperture1"]["diameter"]       # length of the N2F near-field line in y in mm (clipped below)
n2f_nangles = 721                  # angular samples of the far-field pattern over +/-90 deg

n2f_x = near_field_x_mm
if n2f_x < -size_x/2 + pml_thickness:
    n2f_x = -size_x/2 + pml_thickness + 1.0/res
    print(f"WARNING: near-field line was inside the PML, moved to x = {n2f_x:.2f} mm")

# Keep the line monitor out of the top/bottom PMLs
n2f_size_y = min(near_field_diameter, size_y - 2*pml_thickness)

n2f_monitor = simulation.add_near2far(
    fcen, 0, 1,
    mp.Near2FarRegion(center=mp.Vector3(n2f_x, 0),
                      size=mp.Vector3(0, n2f_size_y),
                      weight=-1))
print(f"N2F near-field line monitor: x = {n2f_x:.2f} mm, "
      f"length = {n2f_size_y:.2f} mm (PML thickness = {pml_thickness:.2f} mm)")

# DFT field maps at fcen over the full cell: for a pulsed run these replace the
# CW time-averaged field maps used by the post-simulation analysis (the
# instantaneous fields are ~0 once the pulse has left the domain)
dft_volume = mp.Volume(center=mp.Vector3(0, 0, 0), size=mp.Vector3(size_x, size_y, 0))
dft_fields = stepfunctions.setup_dft_fields(
    simulation, freq=fcen, components=[mp.Ez, mp.Hx, mp.Hy], where=dft_volume)

Now we are all set to run the simulation!!

In [ ]:
# Run until the pulse has left the domain and every DFT monitor (N2F line +
# field maps) has converged, with a hard cap at 2x the previous fixed runtime
simulation.run(mp.at_every(runtime_params["animation_timestep"], stepfunctions.Ez2_dB),
               mp.at_every(runtime_params["dt"], stepfunctions.count_dft_sample),
               mp.at_end(stepfunctions.save_animation),
               mp.at_end(stepfunctions.save_dft_fields),
               mp.at_end(stepfunctions.extract_xyzw),
               until_after_sources=mp.stop_when_dft_decayed(
                   tol=1e-8,
                   maximum_run_time=2*runtime_params["total_time"]))

print("Simulation completed.")

# #~ ---------------------------------------------

# Save the final edited JSON data
with open(data["output"]["savepath"]["path"] + "/" + data["simulation"]["name"] + "_simulation_data.json", "w") as f:
    json.dump(data, f, indent=2)
print(f"Simulation parameters saved to: {data['output']['savepath']['path']}{data['simulation']['name']}_simulation_data.json")

# Post Simulation Analysis

Now we will be extracting the following information from the post-processed data:
1) Extracting the frequency-domain (DFT) fields at `fcen` and verifying the aperture profile against industry standard softwares such as CST and GRASP.
2) Calculating the poynting vector $\textbf{S}$ from the DFT $\textbf{E}$ and $\textbf{H}$ fields.
3) Computing the far-field beam with MEEP's built-in near-to-far field (N2F) transformation and comparing it with GRASP and CST.

First, let's grab the frequency-domain fields at `fcen` from the DFT monitors, save them to disk (`field_dft_fcen.npz`), and load them back — mirroring the file-based workflow of the CW version of this tutorial.

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

def load_fields(basepath, filename):
    # Construct the full path to the file
    filepath = os.path.join(basepath, filename)
    # Load the fields stored in npz files
    data = np.load(filepath)

    return data

# =================== DFT field maps at fcen ===================
ez_dft = simulation.get_dft_array(dft_fields, mp.Ez, 0)
hx_dft = simulation.get_dft_array(dft_fields, mp.Hx, 0)
hy_dft = simulation.get_dft_array(dft_fields, mp.Hy, 0)
(x_coords, y_coords, z_coords, weights) = simulation.get_array_metadata(vol=dft_volume)
print(f"DFT field map shape: {ez_dft.shape}; coords: ({len(x_coords)}, {len(y_coords)})")

np.savez_compressed(
    os.path.join(savepath, "field_dft_fcen.npz"),
    ez_real=np.real(ez_dft), ez_imag=np.imag(ez_dft),
    hx_real=np.real(hx_dft), hx_imag=np.imag(hx_dft),
    hy_real=np.real(hy_dft), hy_imag=np.imag(hy_dft),
    x_coords=np.asarray(x_coords), y_coords=np.asarray(y_coords),
    fcen=fcen)
print(f"DFT field maps at fcen saved to: {os.path.join(savepath, 'field_dft_fcen.npz')}")

basepath = os.path.join(savepath)
dft_data = load_fields(basepath, 'field_dft_fcen.npz')
print("DFT field data keys:", dft_data.files)

# Keep the same coordinate container name the CW version of this tutorial used
xyzw_data = {'x_coords': dft_data['x_coords'], 'y_coords': dft_data['y_coords']}

Now since the 2D simulations are done in TE mode polarization, only the Ez, Hx, Hy components survive. These are the frequency-domain (DFT) fields at the pulse centre frequency `fcen`, so they are complex-valued (amplitude AND phase) just like the CW time-averaged fields were.

In [ ]:
# TE component (Ez, Hx, Hy) at the pulse centre frequency fcen
ez = dft_data['ez_real'] + 1j * dft_data['ez_imag']
hx = dft_data['hx_real'] + 1j * dft_data['hx_imag']
hy = dft_data['hy_real'] + 1j * dft_data['hy_imag']

Now lets calculate the corresponding poynting vector $\textbf{S}$. The Poynting vector components in TE mode are:

$S_x = -E_z \cdot H_y^*$

$S_y = E_z \cdot H_x^*$

In [ ]:
# S vector components for TE mode
sx = -ez * np.conj(hy)
sy = ez * np.conj(hx)
sx_mag = np.abs(sx)
sy_mag = np.abs(sy)
s_total = np.sqrt(sx_mag**2 + sy_mag**2)
s_total_db = 10 * np.log10(s_total / np.max(s_total) + 1e-20)  # in dB
# efield magnitude
ez_power = np.abs(ez)**2
ez_power_db = 10 * np.log10(ez_power / np.max(ez_power) + 1e-20)  # in dB

Let's plot the Poynting vector magnitude and efield power

In [ ]:
def plot_field(simname, field_db, title, filename, xcoords, ycoords, freq,
               vmin=-40, vmax=0, 
               savepath= os.path.join('./../processed_data/'),
                show_plots=True):
    import matplotlib.pyplot as plt
    plt.style.use('default')
    
    plt.figure(figsize=(8, 6))
    plt.imshow(field_db.T, extent=(xcoords[0], xcoords[-1], ycoords[0], ycoords[-1]),
               origin='lower', cmap='inferno', vmin=vmin, vmax=vmax)
    plt.colorbar(label='dB')
    plt.title(title)
    plt.xlabel('x (mm)')
    plt.ylabel('y (mm)')

    if savepath:
        # Create directory with simname and frequency subdirectories
        save_dir = os.path.join(savepath, simname, f'{freq}GHz')
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, filename), dpi=300)
        # Save as a svg file as well for publication
        plt.savefig(os.path.join(save_dir, filename).replace('.png', '.svg'), dpi=300) 
        print(f"{title} plot saved to: {os.path.join(save_dir, filename)}")
    if show_plots:
        plt.show()

# E-field plot
plot_field(simname = 'simple_single_lens_ARC', 
            field_db=ez_power_db, 
            title='E-field Magnitude (dB)', 
            filename='ez_magnitude_db.png', 
            xcoords=xyzw_data['x_coords'], 
            ycoords=xyzw_data['y_coords'],
            freq=freq)
# S-vector plot
plot_field(simname = 'simple_single_lens_ARC', 
            field_db= s_total_db, 
            title='Poynting Vector Magnitude (dB)', 
            filename='s_magnitude_db.png', 
            xcoords=xyzw_data['x_coords'], 
            ycoords=xyzw_data['y_coords'],
            freq=freq)

# Far Field Beam comparision with CST and GRASP!

## Far field via MEEP's N2F transformation

The MeepSAT/MEEP far-field beam is computed with MEEP's built-in near-to-far field transformation: `get_farfield()` evaluates the radiation pattern from the DFT near-field line monitor registered before the run. The far field is evaluated on a semicircle centred on the near-field line, at a radius well beyond the Fraunhofer distance $2D^2/\lambda$ so the angular pattern has converged. In 2D the fields decay as $1/\sqrt{r}$; the pattern is normalised to its peak so the overall scale drops out.

In [ ]:
# =================== Far field via MEEP's N2F transformation ===================
# Evaluate the far field on a semicircle centred on the near-field line, at a
# radius well beyond the Fraunhofer distance 2*D^2/lambda so the angular pattern
# has converged. In 2D the fields decay as 1/sqrt(r); the pattern is normalised
# to its peak so the overall scale drops out.
wvl_mm = 1.0 / fcen  # a = 1 mm  =>  lambda [mm] = 1 / freq [meep units]
fraunhofer_mm = 2.0 * n2f_size_y**2 / wvl_mm
far_field_radius = 10.0 * fraunhofer_mm
print(f"Fraunhofer distance 2D^2/lambda = {fraunhofer_mm:.1f} mm; "
      f"evaluating far fields at r = {far_field_radius:.1f} mm")

# theta is measured from the -x axis (the optical axis), positive towards +y
ff_angles_deg = np.linspace(-90.0, 90.0, int(n2f_nangles))
ff_angles_rad = np.radians(ff_angles_deg)

# get_farfield() returns (Ex, Ey, Ez, Hx, Hy, Hz) at the requested point
ff_fields = np.zeros((len(ff_angles_rad), 6), dtype=np.complex128)
for k, theta in enumerate(ff_angles_rad):
    ff_point = mp.Vector3(n2f_x - far_field_radius*np.cos(theta),
                          far_field_radius*np.sin(theta))
    ff_fields[k, :] = simulation.get_farfield(n2f_monitor, ff_point)

ff_Ex, ff_Ey, ff_Ez = ff_fields[:, 0], ff_fields[:, 1], ff_fields[:, 2]
ff_Hx, ff_Hy, ff_Hz = ff_fields[:, 3], ff_fields[:, 4], ff_fields[:, 5]

# Radial (outward) component of the time-averaged Poynting vector
Sx = np.real(ff_Ey*np.conj(ff_Hz) - ff_Ez*np.conj(ff_Hy))
Sy = np.real(ff_Ez*np.conj(ff_Hx) - ff_Ex*np.conj(ff_Hz))
S_r = np.clip(-Sx*np.cos(ff_angles_rad) + Sy*np.sin(ff_angles_rad), 0.0, None)
ff_power_dB = 10*np.log10(S_r/np.max(S_r) + 1e-20)

# |Ez|^2 pattern as a cross-check (Ez polarisation => S_r ~ |Ez|^2 in the far field)
ff_Ez2 = np.abs(ff_Ez)**2
ff_Ez2_dB = 10*np.log10(ff_Ez2/np.max(ff_Ez2) + 1e-20)

n2f_results = {
    'angle': ff_angles_deg,
    'power': S_r,
    'power_dB': ff_power_dB,
    'Ez_far': ff_Ez,
    'Ez2_dB': ff_Ez2_dB,
    'far_field_radius': far_field_radius,
    'n2f_x': n2f_x,
    'n2f_size_y': n2f_size_y,
    'fcen': fcen}
np.savez(os.path.join(savepath, "n2f_far_field.npz"), **n2f_results)
print(f"N2F far-field results saved to: {os.path.join(savepath, 'n2f_far_field.npz')}")

The MeepSAT far field above is computed with MEEP's **near-to-far field (N2F) transformation** (a rigorous surface-equivalence propagation of the DFT near fields), rather than the simple [Fraunhofer diffraction](https://en.wikipedia.org/wiki/Fraunhofer_diffraction_equation) FFT of the aperture field used in earlier versions of this tutorial.

For the reference softwares:
- **GRASP** provides its own far-field beams (computed inside GRASP via spherical wave expansion), which we load directly from the `far_field_beams_*.h5` files; in addition we FFT its exported aperture field for a Fraunhofer-style cross-check.
- **CST** only exported the aperture field, so its far field is obtained via the Fraunhofer FFT of that aperture slice.

First we will define the various dictionaries and file paths to the GRASP and CST simulations data followed by defining some customisable functions to read the aperture GRASP and CST simulated datasets. 

In [ ]:
import meepsat.field_analysis as analysis

#FOR GRASP
def load_grasp_aperture_data(aperture_file, freq_label):
    print(f"Loading GRASP aperture data from: {aperture_file} for frequency label: {freq_label}")
    with h5py.File(aperture_file, 'r') as f:
        print("Keys in the HDF5 file:", list(f.keys()))
        print(f"Keys in the frequency group '{freq_label}':", list(f[freq_label].keys()))
        aperture_data = {
            'Ex': f[freq_label]['Ex'][:],
            'Ey': f[freq_label]['Ey'][:],
            'Ez': f[freq_label]['Ez'][:],
            'x': f[freq_label]['x'][:],
            'y': f[freq_label]['y'][:],
        }
        print("GRASP resolution (y)", analysis.calculate_grasp_resolution(aperture_data['y']))
    return aperture_data

def load_grasp_farfield_data(farfield_file, freq_label):
    print(f"Loading GRASP far field data from: {farfield_file} for frequency label: {freq_label}")
    with h5py.File(farfield_file, 'r') as f:
        print("Keys in the HDF5 file:", list(f.keys()))
        print(f"Keys in the frequency group '{freq_label}':", list(f[freq_label].keys()))
        Eco = f[freq_label]['Ex'][:]
        Ecx = f[freq_label]['Ey'][:]
        x = f[freq_label]['x'][:]
        
        Eco_dB = 10 * np.log10(np.abs(Eco)**2 / np.max(np.abs(Eco)**2))
        center_y, center_x = np.array(Eco_dB.shape) // 2
        
        return {
            'angle': x,
            'power_dB': Eco_dB[:, center_x],
        }

# FOR CST
def load_cst_data(ey_file, s_file, plot_label = 'CST TE'):
    # Load Ey data by using pandas
    import pandas as pd
    ey_data = pd.read_csv(ey_file, sep=r'\s+', skiprows=3, names=['Y', 'Re(Ey)', 'Im(Ey)'])
    s_mag_data = pd.read_csv(s_file, sep=r'\s+', skiprows=3, names=['Y', 'S_Mag_linear'])

    print(f"file found: {ey_file}, {s_file}")

    data_dict = {
        'efield': ey_data,
        's_mag': s_mag_data,
        'plot_label': plot_label
    }

    return data_dict

# ========================= GRASP aperture field data =============================
base_grasp_data_dir = Path('auxiliary_data/01_simple_single_lens_ARC/GRASP_data/50mm_lens_sim')

"""
Define GRASP files in a structured way for extracting Aperture 
and farfield data (calculated within GRASP using spherical decomposition) 
for different simulation configurations (PO noARC, MoM with ARC, MoM noARC)
"""

grasp_files = {
    'PO_noARC': {
        'aperture': 'apertureField_PO.h5',
        'farfield': 'far_field_beams_PO.h5',
        'label': 'GRASP PO noARC'
    },
    'MoM_ARC': {
        'aperture': 'apertureField_MoM_AR.h5',
        'farfield': 'far_field_beams_MoM_AR.h5',
        'label': 'GRASP MoM with ARC'
    },
    'MoM_noARC': {
        'aperture': 'apertureField_MoM_noAR.h5',
        'farfield': 'far_field_beams_MoM_noAR.h5',
        'label': 'GRASP MoM noARC'
    }
}

# Add full paths
for key in grasp_files:
    grasp_files[key]['aperture'] = str(base_grasp_data_dir / f'50mm_lens_90120150GHz_{grasp_files[key]["aperture"]}')
    grasp_files[key]['farfield'] = str(base_grasp_data_dir / f'50mm_lens_90120150GHz_{grasp_files[key]["farfield"]}')
    

# Load all GRASP data
grasp_aperture_data = {}
grasp_farfield_data = {}
freq_to_analyse = str(int(freq))
c = 2.998e+11  # Speed of light in mm/s
wvl_meep = c / (freq * 1e9)  # Wavelength in mm
print(f"Frequency to analyse: {freq_to_analyse} GHz, corresponding wavelength: {wvl_meep:.2f} mm")

for key, files in grasp_files.items():
    aperture_data = load_grasp_aperture_data(files['aperture'], freq_to_analyse)
    aperture_data['plot_label'] = files['label'].replace('GRASP ', '')
    grasp_aperture_data[key] = aperture_data
    
    farfield_data = load_grasp_farfield_data(files['farfield'], freq_to_analyse)
    farfield_data['plot_label'] = files['label']
    grasp_farfield_data[key] = farfield_data

# ========================= CST APERTURE DATA =============================
# TE Ey file format: 120mm_TE_Ey_ReIm_arc_{}_GHz.txt.format(freq)
# TM Ey file format: 120mm_TM_Ey_ReIm_arc_{}_GHz.txt.format(freq)
# TE S file format: 120mm_TE_S_Mag_arc_{}_GHz.txt.format(freq)
# TM S file format: 120mm_TM_S_Mag_arc_{}_GHz.txt.format(freq)

TE_Ey_data_file = Path('auxiliary_data/01_simple_single_lens_ARC/CST_data/ARC/Field data along both X and Y slices 120mm focus (TM and TE pols)/120mm_TE_Ey_ReIm_arc_{}_GHz.txt'.format(freq_to_analyse))
TE_S_data_file = Path('auxiliary_data/01_simple_single_lens_ARC/CST_data/ARC/Field data along both X and Y slices 120mm focus (TM and TE pols)/120mm_TE_S_Mag_arc_{}_GHz.txt'.format(freq_to_analyse))

CST_data_with_ARC = load_cst_data(TE_Ey_data_file, TE_S_data_file, plot_label='CST TE (with ARC)')


Now we will be calculating the far field beam from the aperture field profile for GRASP and CST (the MeepSAT far field was already computed above with MEEP's N2F transformation)!

In [ ]:
"""
zeropad controls the zero-padding applied to the aperture field 
before computing the far-field beam pattern using FFT (Fast Fourier Transform).

NOTE: with the N2F approach the MeepSAT far field no longer uses this FFT;
zeropad only affects the GRASP / CST aperture-FFT far fields below.

Purpose:
- Increases the frequency resolution in the far-field domain
- Provides smoother interpolation of the far-field radiation pattern
- Higher values give finer angular resolution but increase computation time

How it works:
The aperture field is padded with zeros to (original_length of efield * zeropad) points
before FFT, effectively increasing the number of far-field angle samples.
"""
zeropad = 15

# Aperture radius (in mm)
# Defines the physical extent of the aperture used for far-field calculation
x_pos = data["apertures"]["aperture1"]["pos_x"] #-data["simulation"]["primary_params"]["cell_size"]["x"]/2 + 4 #data["apertures"]["aperture1"]["pos_x"]
aperture_radius =  data["apertures"]["aperture1"]["diameter"]/2#data["simulation"]["primary_params"]["cell_size"]["y"]/2 #data["apertures"]["aperture1"]["diameter"]/2
print(f"Aperture radius: {aperture_radius} mm")

Now lets extract the aperture slice E-field for GRASP MoM, CST FIT and MeepSAT FDTD

In [ ]:
# GRASP (Ex is the dominant component for TE mode, and we are ignoring Ez and Ey for simplicity)
aperture_slice_grasp_mom_ez = 0
aperture_slice_grasp_mom_ey = 0
aperture_slice_grasp_mom_ex = grasp_aperture_data['MoM_ARC']['Ex'][:, grasp_aperture_data['MoM_ARC']['Ex'].shape[1] // 2]
# Keep the slice complex (amplitude AND phase) for the far-field FFT
aperture_slice_grasp_mom = aperture_slice_grasp_mom_ez + aperture_slice_grasp_mom_ex + aperture_slice_grasp_mom_ey
# Set everything else to NaN except the values within the aperture diameter
aperture_slice_grasp_mom = np.where(np.abs(grasp_aperture_data['MoM_ARC']['y']) <= aperture_radius, aperture_slice_grasp_mom, np.nan)
# Remove NaN indices from both the power and the corresponding y-coordinates
valid_indices_grasp = ~np.isnan(aperture_slice_grasp_mom)
aperture_slice_grasp_mom = aperture_slice_grasp_mom[valid_indices_grasp]
y_coords_grasp = grasp_aperture_data['MoM_ARC']['y'][valid_indices_grasp]
aperture_slice_grasp_mom = aperture_slice_grasp_mom/np.max(np.abs(aperture_slice_grasp_mom))
aperture_slice_grasp_mom_dB = 10 * np.log10(np.abs(aperture_slice_grasp_mom) + 1e-20)  # in dB

# CST (Ey is the dominant component for TE mode simulations done with CST)
# Keep the slice complex (amplitude AND phase) for the far-field FFT
aperture_slice_cst = CST_data_with_ARC['efield']['Re(Ey)'] + 1j*CST_data_with_ARC['efield']['Im(Ey)'] #np.conj(CST_data_with_ARC['efield']['Re(Ey)'] + 1j*CST_data_with_ARC['efield']['Im(Ey)'])
# Set everything else to NaN except the values within the aperture diameter
aperture_slice_cst = np.where(np.abs(CST_data_with_ARC['efield']['Y']) <= aperture_radius, aperture_slice_cst, np.nan)
# Remove NaN indices from both the power and the corresponding y-coordinates
valid_indices_cst = ~np.isnan(aperture_slice_cst)
aperture_slice_cst = aperture_slice_cst[valid_indices_cst]
y_coords_cst = CST_data_with_ARC['efield']['Y'][valid_indices_cst]
aperture_slice_cst = np.array(aperture_slice_cst/np.max(np.abs(aperture_slice_cst)))
aperture_slice_cst_dB = 10 * np.log10(np.abs(aperture_slice_cst) + 1e-20)  # in dB


# MeepSAT (Ez is the dominant component for TE mode in MeepSAT simulations)
# Extract a 1D slice at the aperture location (x=-60 mm)
x_index = (np.abs(xyzw_data['x_coords'] - (x_pos))).argmin()
aperture_slice_meepsat = ez[x_index, :]  # Extract 1D slice at x=-60, kept complex (amplitude AND phase)
# Set everything else to NaN except the values within the aperture diameter
aperture_slice_meepsat = np.where(np.abs(xyzw_data['y_coords']) <= aperture_radius, aperture_slice_meepsat, np.nan)
# Remove NaN indices from both the power and the corresponding y-coordinates
valid_indices = ~np.isnan(aperture_slice_meepsat)
aperture_slice_meepsat = aperture_slice_meepsat[valid_indices]
y_coords_meepsat = xyzw_data['y_coords'][valid_indices]
aperture_slice_meepsat = aperture_slice_meepsat/np.max(np.abs(aperture_slice_meepsat))  # Normalize the field for better comparison
aperture_slice_meepsat_dB = 10 * np.log10(np.abs(aperture_slice_meepsat) + 1e-20)  # in dB

### Optional: refocus the CST aperture slice

The CST field was exported at a plane where the beam is not collimated: its wavefront carries a residual quadratic phase (defocus) of about 0.58 waves peak-to-valley across the aperture at 150 GHz (equivalent wavefront radius of curvature ~218 mm), while the MeepSAT and GRASP aperture phases are flat to within a few hundredths of a wave. This defocus broadens the CST far-field main lobe (~7.5 deg vs ~2 deg at -3 dB) and raises its shoulders — an effect that was invisible when only the field magnitude was used.

The cell below fits the quadratic term of the CST aperture phase (over the well-illuminated region) and removes it, so the CST far field can be compared like-for-like with GRASP and MeepSAT. The fit is done per-frequency, so it adapts automatically at 90/120/150 GHz. Set `refocus_cst = False` to keep the raw CST field.

In [ ]:
# Optional: remove the residual quadratic phase (defocus) from the CST aperture slice.
# Set refocus_cst = False to keep the raw CST field.
refocus_cst = True

if refocus_cst:
    cst_phase = np.unwrap(np.angle(aperture_slice_cst))
    cst_y = np.asarray(y_coords_cst, dtype=float)
    # Fit the quadratic only over the well-illuminated part of the aperture
    illuminated = np.abs(aperture_slice_cst) > 0.1
    quad_coeff = np.polyfit(cst_y[illuminated], cst_phase[illuminated], 2)[0]
    # Remove only the quadratic (defocus) term; keep tilt and global phase untouched
    aperture_slice_cst = aperture_slice_cst * np.exp(-1j * quad_coeff * cst_y**2)
    k_wave = 2 * np.pi / wvl_meep
    print(f"Removed quadratic phase from CST slice: {quad_coeff:.5f} rad/mm^2 "
          f"(equivalent wavefront radius of curvature: {k_wave/(2*quad_coeff):.0f} mm)")

Let's plot the profiles in linear scale to visualise

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('default')

# Plot the efield profiles at the aperture for comparision
plt.figure(figsize=(8, 6))

# The slices are complex now, so plot their magnitude
# MeepSAT aperture slice (normalized)
plt.plot(y_coords_meepsat, np.abs(aperture_slice_meepsat), label='MEEPSAT FDTD')
# GRASP MOM aperture slice (normalized)
plt.plot(y_coords_grasp, np.abs(aperture_slice_grasp_mom), label='GRASP MoM')
# CST aperture slice (normalized)
plt.plot(y_coords_cst, np.abs(aperture_slice_cst), label='CST FIT')

plt.axvline(x=aperture_radius, color='blue', linestyle='--', label='Aperture Edge +')
plt.axvline(x=-aperture_radius, color='red', linestyle='--', label='Aperture Edge -')
plt.title('E-field Profile at Aperture')
plt.xlabel('Y Coordinate (mm)')
plt.ylabel('Normalized E-field Magnitude (linear scale)')
plt.grid()
plt.legend()

# Save the figure 
save_dir = os.path.join('./../processed_data', 'simple_single_lens_ARC', f'{freq}GHz')
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'aperture_efield_comparison.png'), dpi=300, bbox_inches='tight')

# Save as a svg file for better quality in publications
plt.savefig(os.path.join(save_dir, 'aperture_efield_comparison.svg'), dpi=300, bbox_inches='tight')

plt.show()

import numpy as np

# Interpolate the normalized field at the aperture edge
E_edge = np.interp(aperture_radius,
                   y_coords_meepsat,
                   np.abs(aperture_slice_meepsat))

ET_dB = 10 * np.log10(E_edge)

print(f"Edge taper = {ET_dB:.2f} dB")

Now let's assemble the far-field beams and visualise the results:

- **MEEPSAT FDTD**: MEEP's native N2F far field computed above (`n2f_results`).
- **GRASP MoM**: both GRASP's own far field (spherical wave expansion, loaded from the h5 file) and the Fraunhofer FFT of its aperture slice via `analysis.meepsat_farfield`.
- **CST FIT**: the Fraunhofer FFT of its aperture slice via `analysis.meepsat_farfield`.

Since the aperture slices are passed as complex arrays (amplitude AND phase), the FFT dictionaries also contain the far-field phase (`phase_rad`); for the N2F result the phase comes from the complex far-field $E_z$. Each dataset has its own arbitrary global phase reference (and the N2F phase is referenced to the near-field line position rather than the aperture plane), so each phase curve is zeroed at boresight before plotting and the phase comparison should be read as indicative.

In [ ]:
# If the simulation was run in a previous session, load the saved N2F results instead
if 'n2f_results' not in globals():
    n2f_results = dict(np.load(os.path.join(savepath, 'n2f_far_field.npz')))

# MeepSAT far field: MEEP's native near-to-far (N2F) transformation computed above
meepsat_farfield_dict = {
    'angle': np.asarray(n2f_results['angle']),
    'power_dB': np.asarray(n2f_results['power_dB']),
    'phase_rad': np.angle(np.asarray(n2f_results['Ez_far'])),
    'complex_farfield': np.asarray(n2f_results['Ez_far']),
    'plot_label': 'MEEPSAT FDTD (MEEP N2F)'}

# GRASP / CST far fields: Fraunhofer FFT of their aperture slices (as before)
grasp_MOM_fft_dict = analysis.meepsat_farfield(efield= aperture_slice_grasp_mom,
             coords= y_coords_grasp,
             wavelength= wvl_meep,
             resolution= analysis.calculate_grasp_resolution(y_coords_grasp),
             zero_pad_beam=zeropad,
             plot_label='GRASP MoM (aperture FFT)')

cst_fit_dict = analysis.meepsat_farfield(efield= aperture_slice_cst,
             coords= y_coords_cst,
             wavelength= wvl_meep,
             resolution= analysis.calculate_grasp_resolution(y_coords_cst),
             zero_pad_beam=zeropad,
             plot_label='CST FIT (aperture FFT)')

# GRASP's own far field, computed inside GRASP via spherical wave expansion
grasp_native_ff_dict = {
    'angle': grasp_farfield_data['MoM_ARC']['angle'],
    'power_dB': grasp_farfield_data['MoM_ARC']['power_dB'],
    'plot_label': grasp_farfield_data['MoM_ARC']['plot_label'] + ' (native far field)'}

import matplotlib.pyplot as plt
plt.style.use('default')

# Plot the farfield patterns for comparison
plt.figure(figsize=(8, 6))
plt.plot(meepsat_farfield_dict['angle'], meepsat_farfield_dict['power_dB'], label= meepsat_farfield_dict['plot_label'])
plt.plot(grasp_native_ff_dict['angle'], grasp_native_ff_dict['power_dB'], label=grasp_native_ff_dict['plot_label'])
plt.plot(grasp_MOM_fft_dict['angle'], grasp_MOM_fft_dict['power_dB'], label=grasp_MOM_fft_dict['plot_label'], linestyle='--')
plt.plot(cst_fit_dict['angle'], cst_fit_dict['power_dB'], label=cst_fit_dict['plot_label'])
plt.xlabel('Angle (degrees)')
plt.ylabel('Power (dB)')
plt.xlim(0, 20)
plt.ylim(-60,0)
plt.legend()

# Save the figure
save_dir = os.path.join('./../processed_data', 'simple_single_lens_ARC', f'{freq}GHz')
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'farfield_comparison.png'), dpi=300, bbox_inches='tight')

# Save as a svg file for better quality in publications
plt.savefig(os.path.join(save_dir, 'farfield_comparison.svg'), dpi=300, bbox_inches='tight')

plt.show()

# Plot the farfield phase for comparison
plt.figure(figsize=(8, 6))
for ff_dict in [meepsat_farfield_dict, grasp_MOM_fft_dict, cst_fit_dict]:
    window = np.abs(ff_dict['angle']) <= 50
    phase_deg = np.degrees(np.unwrap(ff_dict['phase_rad'][window]))
    # Each dataset has an arbitrary global phase reference, so zero the phase at boresight
    phase_deg -= phase_deg[np.argmin(np.abs(ff_dict['angle'][window]))]
    plt.plot(ff_dict['angle'][window], phase_deg, label=ff_dict['plot_label'])
plt.xlabel('Angle (degrees)')
plt.ylabel('Far-field Phase (degrees, unwrapped)')
# plt.xlim(-50, 50)
plt.grid()
plt.legend()

# Save the figure
plt.savefig(os.path.join(save_dir, 'farfield_phase_comparison.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(save_dir, 'farfield_phase_comparison.svg'), dpi=300, bbox_inches='tight')

plt.show()

If for some reason, you want to run it using mpirun and bash script, there is an example provided [here](https://github.com/aa16oaslak/MeepSAT/tree/main/examples/HPC_Tutorial/time-reverse/01_simple_single_lens_ARC)